In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
# modify global setting for plotting

import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 3
mpl.rcParams['font.family'] = 'Arial'

### color maps ###

In [ ]:
m_color = {'L2a':'olive', 'L2b':'olivedrab', 'L2c':'yellowgreen', 'L3a':'goldenrod','L3b':'gold', 
           'L4a':'steelblue', 'L4b': 'cornflowerblue', 'L4c':'lightskyblue', 
           'L5a':'#663399', 'L5b':'mediumpurple', 
           'L5b-1':"#F506D1", 'L5b-2': "#BD7CC8",'L5b-s':"#FC87F6", 
           'L5ET':'lightseagreen', 'L5NP':'palevioletred',
             'L6short-a':'mediumvioletred', 'L6short-b':'violet', 
             'L6tall-a':'firebrick', 'L6tall-b':'orangered', 'L6tall-c':'salmon',
             'L6wm':'darkgrey', 
             'DTC':"#EE0303",'PTC':"#0209DC",'STC':'orange','ITC':'green', 
             np.nan: 'lightgrey'} 
lv1_EI_color = {'excitatory_neuron':'mediumvioletred', 'inhibitory_neuron':'teal'}
lv2_EI_color = {'excitatory_neuron':'indigo', 'inhibitory_neuron':'steelblue'}

grouped_m_color = {'L2':"#83BD0F", 'L3':"#FFD500",'L4':"#0085BA", 'L5IT':'#663399', 'L5ET': 'lightseagreen', 'L6':'firebrick', 'DTC':'red', 'PTC':'blue', 'STC':'orange', 'ITC':'green'}

cmap_E = sns.light_palette("#C71585", as_cmap=True)
cmap_I = sns.light_palette("teal", as_cmap=True)


### Load data ###

In [ ]:
path = '.../Data files/'

In [ ]:
# Proofread excitatory neuron info
prf_cells = pd.read_csv(path+'Proofread IT and ET with root id-v1718.csv')
# Proofread neuron input and output synapse statistics
syn_stat = pd.read_csv(path+'synapse_identification_stat_v1718.csv')
# VISp neuron mtype table
mtype_V1 = pd.read_parquet(path+'V1_mtype_v1718.parquet')
# Column neuron mtype table
mtype_col = pd.read_csv(path+'Column_mtype_v1718.csv')
# Motif table
motif_df = pd.read_csv(path+'prf-inh-motif-sorted-v1664.csv')
# Proofread excitatory neuron output synapses with mtypes
syn_DF_m = pd.read_parquet(path+'Proofread-V1-output-synapses-with-mtypes-v1718.parquet')
# Proofread excitatory neuron output synapses with motifs
syn_motif_DF_m = pd.read_csv(path+'Proofread-V1-output-synapses-with-motif-v1718.csv')
# VISp proofread inhibitory neuron connection table
all_pr_I_conn = pd.read_parquet(path+'V1_prf_inh_connections_v1718.parquet')

### SI-Column cell output scatter plot ###

In [ ]:
col_conn = pd.read_parquet(path+'Column-all-output-v1718-mtype.parquet')

In [ ]:
col_conn=col_conn.merge(mtype_V1[['target_id', 'cell_type']], left_on='id_post', right_on='target_id', how='left', suffixes=('_old', '_new'))
col_conn.drop(columns=['target_id', 'cell_type_post'], inplace=True)
col_conn.rename(columns={'cell_type':'cell_type_post'}, inplace=True)

In [ ]:
E_conn = col_conn.query('cell_type_post.notna()') 
# get ordered id list for plotting
column_order = desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a', 'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b', 'L6wm']

E_conn['cell_type_pre'] = pd.Categorical(
    E_conn['cell_type_pre'],
    categories=column_order,
    ordered=True
)
pre_ids = E_conn.sort_values(['cell_type_pre','pt_position_y_pre'], ascending=True).id_pre.unique().tolist()

desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
        'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b','DTC', 'PTC', 'STC', 'ITC']

E_conn['cell_type_post'] = pd.Categorical(
    E_conn['cell_type_post'],
    categories=desired_order,
    ordered=True
)
post_df = E_conn.groupby('id_post').agg({
    'cell_type_post': 'first',
    'pt_position_y_post': 'first'  
}).reset_index()

post_ids = post_df.sort_values(
    ['cell_type_post', 'pt_position_y_post']
)['id_post'].tolist()

In [ ]:
E_conn_pivot = E_conn.pivot_table(index='id_post', columns='id_pre', values='n_syn', aggfunc='sum', fill_value=0)
E_conn_pivot = E_conn_pivot.reindex(index=post_ids, columns=pre_ids)
E_conn_pivot = E_conn_pivot.astype(int)

In [ ]:
#generate long format for plotting
E_conn_melt = E_conn_pivot.reset_index().melt(id_vars='id_post', var_name='id_pre', value_name='n_syn')
E_conn_melt[['id_post', 'id_pre']] = E_conn_melt[['id_post','id_pre']].astype(str)

In [ ]:
#get coordinates for plotting cell types lines
pre_counts = mtype_col.cell_type.value_counts().reindex(column_order)
pre_coords = pre_counts.cumsum()
post_counts = post_df.sort_values(
    ['cell_type_post', 'pt_position_y_post']).value_counts('cell_type_post').reindex(desired_order)
post_coords = post_counts.cumsum()

In [ ]:

Fig, ax = plt.subplots(figsize=(25,35), dpi=300)
sns.scatterplot(data=E_conn_melt[:], x='id_pre', y='id_post', 
                size='n_syn', sizes=(0, 50), #80
                size_norm=(0, 15), #1000
                #hue='cell_type_IT', palette=m_color,
                #hue_norm=(1, 5), #1000
                color='black', 
                legend=True, ax=ax,
                edgecolor='white', 
                linewidth=0.2,
                alpha=0.5)
for i, x in enumerate(pre_coords - 0.5):
    ax.vlines(x, -1, len(post_ids), color='black', lw=1.5,
              linestyle='dashed', zorder=10)
    
    left = 0 if i == 0 else pre_coords[i - 1]
    text_pos = (left + x) / 2
    ax.text(text_pos, -200, column_order[i], ha='center', va='bottom',
            rotation=45,
            fontsize=18, color='black')

for i, y in enumerate(post_coords - 0.5):
    ax.hlines(y, -2, len(pre_ids), color='black', lw=1.5,
              linestyle='dashed', 
              zorder=10)
    left = 0 if i == 0 else post_coords[i - 1]
    text_pos = (left + y) / 2
    ax.text(-10, text_pos, desired_order[i], ha='right', va='center',
            fontsize=18, color='black')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), title='n_synapses')
ax.axis('off')
#plt.savefig('Column-EtoAll.tiff', format='tiff', dpi=300, bbox_inches='tight')

### Skeleton plot ###

In [ ]:
# Initialize a client for the datastack.
from caveclient import CAVEclient
client = CAVEclient('minnie65_public', version =1718)
from meshparty import skeleton
import skeleton_plot as skelplot
from standard_transform import minnie_ds, minnie_transform_nm
minnie_streamline = minnie_ds.streamline_nm

In [ ]:
nuc_df = client.materialize.tables.nucleus_detection_v0().query()

In [ ]:
nuc_id = 267193
root_id = nuc_df.query(f'id == {nuc_id}').pt_root_id.values[0]

sk_dict = client.skeleton.get_skeleton(root_id, output_format='dict')
#convert skeleton dict to meshparty skeleton 
sk = skeleton.Skeleton.from_dict(sk_dict)
#convert vertices to standard transformed coordinates
sk.vertices= minnie_transform_nm().apply(sk_dict['vertices'])

In [ ]:
#plot the skeleton
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

Layers=np.array([91.80615154, 261.21908419, 396.8631847 , 537.04973966, 753.58049474])

x = Layers
ax.hlines(x, 0, 1400, color='grey', alpha=0.5, linestyle='dashed', lw=1)

skelplot.plot_tools.plot_skel(sk=sk,
                                   invert_y=True,
                                   pull_compartment_colors=True,
                                   pull_radius=False,
                                   plot_soma=True,
                                   soma_size=40,
                                   line_width=1.2,
                                   skel_color_map = { 3:"#7F006C" ,4: "salmon",2: "slategrey" ,1:"#7F006C"  },
                                   ax=ax)  
plt.xlim([300, 1400])
plt.ylim([800, 0])
ax.set_axis_off()   

### Figure 1 ###

#### Figure 1d ####

In [ ]:
#Figure 1d

mtypes = ['L2a','L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 
       'L4c', 'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a', 'L6tall-b', 
       'L6tall-c', 'L6short-a', 'L6short-b','L6wm',
       'PTC', 'DTC', 'STC','ITC' ]
V1_values = pd.DataFrame(mtype_V1.query('cell_type in @mtypes').cell_type.value_counts())
V1_percent = V1_values/V1_values.sum()*100
column_values = pd.DataFrame(mtype_col.query('cell_type in @mtypes').cell_type.value_counts())
column_percent = column_values/column_values.sum()*100
V1_column_p = pd.DataFrame(index=mtypes)
V1_column_p = V1_column_p.join(V1_percent.join(column_percent, lsuffix='_V1', rsuffix='_column'))

data1 = V1_column_p[['count_column']]
data2 = -V1_column_p[['count_V1']]

fig, ax = plt.subplots(figsize=(4, 5), dpi=300)                
sns.barplot(data=data1.T, orient='h', ax=ax,  palette=m_color)
sns.barplot(data=data2.T, orient='h', ax=ax, palette=m_color)

plt.axvline(0, color='white', linewidth=3)  # Place y-axis in the middle

plt.xlim(-15.5,15)
ax.set_xlabel('% of total cell count', fontsize=14, fontname='Arial')
ax.set_ylabel('Mtype', fontsize=14, fontname='Arial')

for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False) 

#### Figure 1e ####

In [ ]:
syn_stat_sub = syn_stat[['cell_type_IT', 'p_all_identified','p_v1_identified']]
syn_stat_melt = syn_stat_sub.melt(id_vars = 'cell_type_IT', var_name='Group', value_name='Percentage')

c_map = {'p_v1_identified':"#FFFFFF", 'p_all_identified':"#737475"}

fig, ax = plt.subplots(figsize=(4, 3), dpi=300)

sns.barplot(data=syn_stat_melt.query('cell_type_IT != "L5ET"'), x='cell_type_IT', y='Percentage', errorbar='se', 
            hue='Group', 
            err_kws={'linewidth': 1}, ax=ax, alpha=1, 
            edgecolor="#5C5E5F", linewidth=1,
            palette=c_map)


ax.set_ylim(0, 100)
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)

ax.set_xlabel('cell type', fontsize=12)
ax.set_ylabel('% of output synapses', fontsize=12)

ax.set_title('Percentage of mtype-identified postsynapses', fontsize=15, y=1.05)
plt.legend(loc='upper right', bbox_to_anchor=(1, 1.05))

### Figure 2 and extended Figure 1-3 ###

#### Output connectivity ####

##### heatmap-lv1 #####

In [ ]:
# plot heatmap of synapse percentage
data = syn_DF_m.query('cell_type_IT != "L5ET"')
data = data.merge(mtype_V1[['target_id','pt_position_y']], left_on='Nucleus_ID', right_on='target_id', how='left', suffixes=('_target','_nuc'))
data = data.sort_values(by=['cell_type_IT','pt_position_y_nuc'], ascending=True)

id_order = data.Nucleus_ID.unique()
data_p = syn_DF_m.pivot_table(index='cell_type', columns='Nucleus_ID', values='id_syn', aggfunc='count').fillna(0)
desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC', 'ITC']
data_p = data_p.reindex(desired_order)
data_p = data_p.reindex(columns=id_order)
data_percent = data_p/data_p.sum(axis=0) * 100

#to get the lines separate subtypes for heatmap
cell_count = pd.DataFrame(prf_cells.query('cell_type_grouped in ["L2", "L3","L5a", "L5b"]').cell_type_IT.value_counts())
cell_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L5a', 'L5b-1', 'L5b-2', 'L5b-s']
cell_count = cell_count.reindex(cell_order)

coords = cell_count.iloc[:,0]

In [ ]:
#Figure 2c, 4d
data_percent = data_percent.astype(float)
fig, ax = plt.subplots(figsize=(25, 6), dpi=300)

sns.heatmap (data_percent, cmap='viridis', vmax=20, ax=ax)

for i,n in enumerate(coords):
    x = coords[:i+1].sum()
    x2 = coords[:i].sum() 
    ax.vlines(x, 0, 25, color='white', lw=1.5)
    ax.text(x2+(x-x2)/2, -1.8, f'{coords.index[i]}', ha='center', va='top', fontsize=30, color='black',  clip_on=False )
#plt.savefig( dpi=300, bbox_inches='tight')
    

In [ ]:
# to plot the heatmap with motifs Figure 2g,5c
data = syn_DF_m.query('cell_type_IT != "L5ET"')
data = data.merge(mtype_V1[['target_id','pt_position_y']], left_on='Nucleus_ID', right_on='target_id', how='left', suffixes=('_target','_nuc'))
data = data.sort_values(by=['cell_type_IT','pt_position_y_nuc'], ascending=True)

id_order = data.Nucleus_ID.unique()
syn_motif_DF_g = syn_motif_DF_m.groupby(['cell_type_IT','Nucleus_ID','cluster']).agg({'id_syn': 'count','size':'mean'}).reset_index()
syn_motif_DF_g.fillna(0, inplace=True)
syn_motif_DF_g.sort_values(by=['cell_type_IT','Nucleus_ID'], ascending=True, inplace=True)
syn_motif_p = syn_motif_DF_g.pivot_table(index='cluster', columns='Nucleus_ID', values='id_syn', aggfunc='sum')
syn_motif_p.fillna(0, inplace=True)
syn_motif_per = syn_motif_p/syn_motif_p.sum()*100
syn_motif_per = syn_motif_per.reindex(columns=id_order)

#to get the lines separate subtypes for heatmap
cell_count = pd.DataFrame(prf_cells.query('cell_type_grouped in ["L2", "L3","L5a","L5b"]').cell_type_IT.value_counts())
cell_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L5a', 'L5b-1', 'L5b-2', 'L5b-s']
cell_count = cell_count.reindex(cell_order)

coords = cell_count.iloc[:,0]

In [ ]:
syn_motif_per = syn_motif_per.astype(float) 
fig, ax = plt.subplots(figsize=(30, 10))
sns.heatmap (syn_motif_per, cmap='viridis', vmax=50)

for i,n in enumerate(coords):
    x = coords[:i+1].sum()
    x2 = coords[:i].sum() 
    ax.vlines(x, 0, 25, color='white', lw=1.5)
    ax.text(x2+(x-x2)/2, -1.6, f'{coords.index[i]}', ha='center', va='top', fontsize=30, color='black')

##### Bar graph-lv1 #####

In [ ]:
#Extended Figure 1c, 4d
syn_DF_g = syn_DF_m.groupby(['cell_type_IT','Nucleus_ID','cell_type']).agg({'id_syn': 'count', 'post_pt_root_id':'nunique', 'size':'mean'}).reset_index()
syn_DF_g.fillna(0, inplace=True)
syn_DF_g.sort_values(by=['cell_type_IT','Nucleus_ID'], ascending=True, inplace=True)

desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC', 'ITC',]

#mtype
post_dic = {}
for type in ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L5a', 'L5b-1', 'L5b-2']:
    data_subset= syn_DF_g.query(f'cell_type_IT == "{type}"').pivot(index='Nucleus_ID', columns='cell_type', values='id_syn')
    data_subset=data_subset[desired_order]
    post_dic [type] = data_subset

In [ ]:
f, ax =plt.subplots(2, 4, figsize=(15,10), facecolor='white', sharey=True,dpi=300)

ax = ax.flatten()

for i, (key, values) in enumerate(post_dic.items()): 

    ax[i].set_xlabel('synapse number', fontsize=12)
    ax[i].set_ylabel('mtype', fontsize=12)
    for pos in ['top', 'right']:
        ax[i].spines[pos].set_visible(False)
    
    sns.barplot(ax=ax[i], data=values, width=0.75, errorbar="se", 
            err_kws={'linewidth':1.5}, orient='y', palette=m_color, alpha=0.8)
    sns.swarmplot(data=values, ax=ax[i], orient= 'y',  palette=m_color, size=2, alpha=1)

    ax[i].set_title(key, y=1.05, loc='center', fontsize=16, fontname='Arial')
    ax[i].set_xlim([0, 150])
    ax[i].set_xticks([0, 30, 60, 90, 120, 150])
    ax[i].tick_params(axis='both', which='major', labelsize=12, size=8)

plt.subplots_adjust(wspace=0.9, hspace=0.5)

##### syn per conn #####

In [ ]:
syn_DF_g['syn_per_conn'] = syn_DF_g['id_syn']/syn_DF_g['post_pt_root_id']
post_dic_2 = {}
for type in ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L5a', 'L5b-1', 'L5b-2']:
    data_subset= syn_DF_g.query(f'cell_type_IT == "{type}"').pivot(index='Nucleus_ID', columns='cell_type', values='syn_per_conn')
    data_subset=data_subset[desired_order]
    post_dic_2 [type] = data_subset

In [ ]:

f, ax =plt.subplots(2, 4, figsize=(10,10), facecolor='white', sharey=True, dpi=300)

ax = ax.flatten()

for i, (key, values) in enumerate(post_dic_2.items()):

    ax[i].set_xlabel('n_syn/connection', fontsize=12)
    ax[i].set_ylabel('mtype', fontsize=12)
    for pos in ['top', 'right']:
        ax[i].spines[pos].set_visible(False)
    
    sns.pointplot(data=values, ax=ax[i], 
                   orient= 'y',  
                   markersize=6,
                    errorbar=('sd'), capsize=0.3, 
                   palette=m_color, 
                   err_kws={'linewidth': 1},
                   alpha=1)
    ax[i].set_title(key, y=1.05, loc='center', fontsize=16, fontname='Arial')
    ax[i].set_xlim([0.5, 2])

plt.subplots_adjust(wspace=0.7, hspace=0.5)

##### lv2 connectivity #####

In [ ]:
#Get lv2 connectivity 
syn_DF_sorted = syn_DF_m.groupby(['Nucleus_ID','target_id']).agg({'cell_type_IT':'min','cell_type':'min','id_syn':'count', 'size':'sum', 'pt_position_y':'min'}).rename(columns={'size':'size_syn'})
syn_DF_sorted.reset_index(inplace=True)

PTC_ids = syn_DF_sorted.query('cell_type == "PTC"').target_id.to_list()
DTC_ids = syn_DF_sorted.query('cell_type == "DTC"').target_id.to_list()

desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
      'L5a', 'L5b',  'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c', 'L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC','ITC']

data = prf_cells.copy()
V1_prf_ids = all_pr_I_conn['pre_nuc_id'].unique().tolist()
Inh_df = syn_DF_sorted.query('target_id in @V1_prf_ids')

Starter_inh = Inh_df.query('Nucleus_ID in @data.Nucleus_ID') 
secondary_df = Starter_inh.merge(all_pr_I_conn,left_on='target_id', right_on='pre_nuc_id', how='inner')
secondary_df['weighted_syn'] = secondary_df['n_syn'] * secondary_df['id_syn']
IT_syn_2nd_DF = secondary_df.merge(mtype_V1, left_on='post_nuc_id', right_on='target_id', how='inner', suffixes=('_I','_secondary'))

##### Heatmap-lv2 #####

In [ ]:
# Extended Figure 3, 4f
ct = 'L2a'

sub_2nd = IT_syn_2nd_DF.query(f'cell_type_IT == "{ct}"')
DTC_df = sub_2nd.query (f'pre_nuc_id in {DTC_ids}')
PTC_df = sub_2nd.query (f'pre_nuc_id in {PTC_ids}')

DTC_ordered_ids = DTC_df.sort_values(by='pt_position_y_I', ascending=True).pre_nuc_id.unique()
PTC_ordered_ids = PTC_df.sort_values(by='pt_position_y_I', ascending=True).pre_nuc_id.unique()
DTC_pos_y = DTC_df.sort_values(by='pt_position_y_I', ascending=True).pt_position_y_I.to_frame()
PTC_pos_y = PTC_df.sort_values(by='pt_position_y_I', ascending=True).pt_position_y_I.to_frame()

DTC_htm = DTC_df.pivot_table(index='cell_type_secondary', columns='pre_nuc_id', values='weighted_syn', aggfunc='sum', 
                              fill_value=0)

DTC_htm = DTC_htm.reindex(desired_order)
DTC_htm = DTC_htm[DTC_ordered_ids]

PTC_htm = PTC_df.pivot_table(index='cell_type_secondary', columns='pre_nuc_id', values='weighted_syn', aggfunc='sum', 
                              fill_value=0)
PTC_htm = PTC_htm.reindex(desired_order)
PTC_htm = PTC_htm[PTC_ordered_ids]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), sharey=True, dpi=300)
gs = gridspec.GridSpec(2, 2, 
                       width_ratios=[1, 1],   
                       height_ratios=[1, 20])
ax = [plt.subplot(gs[0, 0]), plt.subplot(gs[0, 1]), plt.subplot(gs[1, 0]), plt.subplot(gs[1, 1])]
sns.heatmap(DTC_pos_y.T, ax=ax[0], cmap='viridis', vmin=0, vmax=700, cbar=False)
sns.heatmap(PTC_pos_y.T, ax=ax[1], cmap='viridis', vmin=0, vmax=700, cbar=False)
sns.heatmap(DTC_htm, ax=ax[2], cmap='Reds', vmax=25000, cbar=False)
sns.heatmap(PTC_htm, ax=ax[3], cmap='Blues', vmax=25000, cbar=False)
for i in range(4):
    ax[i].set_axis_off()

ax[0].set_title(f'{ct} dysynaptic DTC connectivity', fontsize=15, y=1.02)
ax[1].set_title(f'{ct} dysynaptic PTC connectivity', fontsize=15, y=1.02)

plt.subplots_adjust(wspace=0.1, hspace=0.1)


##### Bar graph-lv2 #####

In [ ]:
IT_syn_2nd_DF.sort_values(by=['cell_type_secondary'], ascending=True, inplace=True)
IT_syn_2nd_DF_g = IT_syn_2nd_DF.groupby(['Nucleus_ID', 'cell_type_secondary']).agg({'weighted_syn':'sum'})
pivot_df = IT_syn_2nd_DF_g.pivot_table(index='Nucleus_ID', columns='cell_type_secondary', values='weighted_syn', fill_value=0)
pivot_df = pivot_df[desired_order]

pivot_df_percent = pivot_df.div(pivot_df.sum(axis=1), axis=0)*100

In [ ]:
# Extended Figure 3, 4f
ct = "L2a"
data_subset = pivot_df_percent.query(f'Nucleus_ID in @prf_cells.query("cell_type_IT == \'{ct}\'").Nucleus_ID')
fig, ax = plt.subplots(figsize=(4, 5), dpi=300)
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)
sns.barplot(data=data_subset, ax=ax,  orient='y', errorbar='se', palette=m_color, alpha=0.6)
sns.swarmplot(data=data_subset, ax=ax, orient= 'y',  palette=m_color, size=3.5, alpha=1)
#sns.barplot(data=IT_syn_2nd_DF, ax=ax, x='weighted_syn', y='cell_type', orient='h', palette=m_color)
ax.set_xlabel('% of weighted synapse', fontsize=12)
ax.set_ylabel('mtype', fontsize=12)
ax.set_title(f'{ct}', fontsize=15, y=1.05)
ax.set_xlim(0, 35)

#### Depth distribution ####

In [ ]:
# for plotting the depth distribution of postsynaptic targets of each cell type
# lv1 
cell_type = ["L2a"]
bin_range = (0, 750)
num_bins = 75
bins = np.linspace(bin_range[0], bin_range[1], num_bins)
index = bins[:-1]
percentage_E_df_1st = pd.DataFrame()
percentage_I_df_1st = pd.DataFrame()

data = prf_cells.query(f'cell_type_IT in {cell_type}') 

for i, row in data.iterrows():
    # to get just column targets  & target_id in @all_pr_I_conn.pre_nuc_id')
    # to get specific cell types  & cell_type == ""
    dataset_E_1st = syn_DF_m.query(f'Nucleus_ID == {row.Nucleus_ID} & classification_system == "excitatory_neuron" ')   
    dataset_I_1st = syn_DF_m.query(f'Nucleus_ID == {row.Nucleus_ID} & classification_system == "inhibitory_neuron"')
   
  
    subset_E_1st = np.array(dataset_E_1st['pt_position_y'])
    subset_I_1st = np.array(dataset_I_1st['pt_position_y'])
    histogram_E_1st = np.histogram(subset_E_1st, bins=bins)[0]
    his_E = pd.DataFrame(histogram_E_1st, index=index, columns=['post_target'])
    his_E_p = his_E['post_target'] / his_E['post_target'].sum() * 100
    percentage_E_df_1st= pd.concat([percentage_E_df_1st, his_E_p], axis=0)
    histogram_I_1st = np.histogram(subset_I_1st, bins=bins)[0]
    his_I = pd.DataFrame(histogram_I_1st, index=index, columns=['post_target'])
    his_I_p = his_I['post_target'] / his_I['post_target'].sum() * 100
    percentage_I_df_1st = pd.concat([percentage_I_df_1st, his_I_p], axis=0)

percentage_E_df_1st.reset_index(inplace=True)
percentage_E_df_1st.rename(columns={'index': 'depth'}, inplace=True)
percentage_I_df_1st.reset_index(inplace=True)
percentage_I_df_1st.rename(columns={'index': 'depth'}, inplace=True)

In [ ]:
# lv2
bin_range = (0, 750)
num_bins = 75
bins = np.linspace(bin_range[0], bin_range[1], num_bins)
index = bins[:-1]
percentage_E_df_2nd = pd.DataFrame()
percentage_I_df_2nd = pd.DataFrame()

data = prf_cells.query(f'cell_type_IT in {cell_type}') 
for i, row in data.iterrows():
    dataset_E_2nd = IT_syn_2nd_DF.query(f'Nucleus_ID == {row.Nucleus_ID} & classification_system == "excitatory_neuron"')           
    dataset_I_2nd = IT_syn_2nd_DF.query(f'Nucleus_ID == {row.Nucleus_ID} & classification_system == "inhibitory_neuron"')   
    subset_E_2nd = np.array(dataset_E_2nd['pt_position_y_secondary'])
    subset_I_2nd = np.array(dataset_I_2nd['pt_position_y_secondary'])
    E_weight = np.array(dataset_E_2nd['weighted_syn'])
    I_weight = np.array(dataset_I_2nd['weighted_syn'])
    histogram_E_2nd = np.histogram(subset_E_2nd, bins=bins, weights=E_weight)[0]
    his_E = pd.DataFrame(histogram_E_2nd, index=index, columns=['post_target'])
    his_E_p = his_E['post_target'] / his_E['post_target'].sum() * 100
    percentage_E_df_2nd= pd.concat([percentage_E_df_2nd, his_E_p], axis=0)
    histogram_I_2nd = np.histogram(subset_I_2nd, bins=bins, weights=I_weight)[0]
    his_I = pd.DataFrame(histogram_I_2nd, index=index, columns=['post_target'])
    his_I_p = his_I['post_target'] / his_I['post_target'].sum() * 100
    percentage_I_df_2nd = pd.concat([percentage_I_df_2nd, his_I_p], axis=0)


percentage_E_df_2nd.reset_index(inplace=True)
percentage_E_df_2nd.rename(columns={'index': 'depth'}, inplace=True)
percentage_I_df_2nd.reset_index(inplace=True)
percentage_I_df_2nd.rename(columns={'index': 'depth'}, inplace=True)


In [ ]:
#depth distribution plot, change data and colormap to plot different combinations

Layers=np.array([91.80615154, 261.21908419, 396.8631847 , 537.04973966, 753.58049474])

fig, ax = plt.subplots(figsize=(2, 4), dpi=300)
sns.lineplot(data=percentage_E_df_1st,  x='post_target', y='depth', color='mediumvioletred', linewidth=0.7, errorbar='se', orient='y')
sns.lineplot(data=percentage_I_df_1st,  x='post_target', y='depth', color='teal', linewidth=0.7, errorbar='se', orient='y')
sns.scatterplot(data=data, x=13, y='pt_position_y', hue='cell_type_IT', 
                palette=m_color, edgecolor='black', s=20, alpha=0.7,
                legend=False)

x = Layers
ax.hlines(x, 0, 16, color='grey', alpha=0.5, linestyle='dashed', lw=0.8)

ax.set_xlim(0, 15)
ax.set_ylim(800, 0)

ax.set_ylabel('Soma depth (um)', fontsize=10)
ax.set_xlabel('percentage of synapses', fontsize=10)
ax.set_title(f'{cell_type[0]} lv1 E-I targets distribution', fontsize=12, y=1.1, loc='center')

for pos in ['top', 'right']:
    ax.spines[pos].set_visible(False)

In [ ]:
# PTC-DTC posttarget comparison

cell_type = "L5b-2"
bin_range = (0, 750)
num_bins = 75
bins = np.linspace(bin_range[0], bin_range[1], num_bins)
index = bins[:-1]
histograms_1 = pd.DataFrame()
histograms_2 = pd.DataFrame()

data = prf_cells.query(f'cell_type_IT == "{cell_type}" ') 
for i, row in data.iterrows():
    dataset_1 = IT_syn_2nd_DF.query(f'Nucleus_ID == {row.Nucleus_ID} & pre_nuc_id in {DTC_ids} & classification_system == "excitatory_neuron"  ') #         
    dataset_2 = IT_syn_2nd_DF.query(f'Nucleus_ID == {row.Nucleus_ID} & pre_nuc_id in {PTC_ids} & classification_system == "excitatory_neuron" ')  #
    subset_1 = np.array(dataset_1['pt_position_y_secondary'])
    subset_2 = np.array(dataset_2['pt_position_y_secondary'])
    weight_1 = np.array(dataset_1['weighted_syn'])
    weight_2 = np.array(dataset_2['weighted_syn'])
    
    histogram_1 = np.histogram(subset_1, bins=bins, weights=weight_1)[0] 
    his_1 = pd.DataFrame(histogram_1, index=index, columns=['post_target'])
    his_1_p = his_1['post_target'] / his_1['post_target'].sum() * 100
    histograms_1 = pd.concat([histograms_1, his_1_p], axis=0)
   
    histogram_2 = np.histogram(subset_2, bins=bins,weights=weight_2)[0] 
    his_2 = pd.DataFrame(histogram_2, index=index, columns=['post_target'])
    his_2_p = his_2['post_target'] / his_2['post_target'].sum() * 100
    histograms_2 = pd.concat([histograms_2, his_2_p], axis=0) 
    

histograms_1.reset_index(inplace=True)
histograms_1.rename(columns={'index': 'depth'}, inplace=True)
histograms_2.reset_index(inplace=True)
histograms_2.rename(columns={'index': 'depth'}, inplace=True)

In [ ]:
Layers=np.array([91.80615154, 261.21908419, 396.8631847 , 537.04973966, 753.58049474])

fig, ax = plt.subplots(figsize=(4, 8), dpi=300)

sns.lineplot(data=histograms_1,  x='post_target', y='depth', color=m_color['DTC'], errorbar='se', orient='y')

sns.lineplot(data=histograms_2,  x='post_target', y='depth', color=m_color['PTC'], errorbar='se', orient='y')

sns.scatterplot(data=data, x=7, y='pt_position_y', hue='cell_type_IT', 
                palette=m_color, edgecolor='black', s=80, alpha=0.7,
                legend=False)

ax.hlines(x, 0, 16, color='grey', alpha=0.5, linestyle='dashed', lw=1)

ax.set_xlim(0, 8)
ax.set_ylim(800, 0)

ax.set_ylabel('Soma depth (um)', fontsize=12)
ax.set_xlabel('percentage of weighted synapses', fontsize=12)


ax.set_title(f'{cell_type} lv1 Inb output connection distribution', fontsize=15, y=1.05, loc='center')

for pos in ['top', 'right']:
    ax.spines[pos].set_visible(False)

#### input and output synapses ####

In [ ]:
#Extended Figure 1a-top

L23_input_g = pd.read_csv(path + 'Column-L23-input-synapse-number-v1718.csv')

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
sns.scatterplot(data=L23_input_g, x='n_syn', y='pt_position_y', 
                hue='cell_type', palette=m_color, ax=ax, edgecolor='white', s=15, alpha=0.9, 
                legend=False)

ax.set_xlim(500, 12000)
ax.set_ylim(300, 50)

for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)

In [ ]:
# Extended Figure 1a-bottom

syn_DF = syn_stat.copy()

f, ax =plt.subplots(1,2, figsize=(12,3), facecolor='white', dpi=300)

for pos in ['right', 'top']:
    ax[0].spines[pos].set_visible(False)
    ax[1].spines[pos].set_visible(False)

sns.swarmplot(ax = ax[0], data=syn_DF.query('cell_type_IT != "L5ET"'), x='cell_type_IT', y='n_syn_input', hue= 'cell_type_IT',
                palette=m_color, legend=False, s=6, alpha=1, zorder=1)
sns.pointplot(ax = ax[0],
    data=syn_DF.query('cell_type_IT != "L5ET"'), x='cell_type_IT', y='n_syn_input', linestyle='none', linewidth = 1,
   errorbar=('se'), 
    capsize=0.2, color="black", zorder = 2)
sns.swarmplot(ax = ax[1], data=syn_DF.query('cell_type_IT != "L5ET"'), x='cell_type_IT', y='n_syn_vis', hue='cell_type_IT',
                palette=m_color, legend=False, s=6, alpha=1, zorder=1)
sns.pointplot(ax = ax[1],
    data=syn_DF.query('cell_type_IT != "L5ET"'), x='cell_type_IT', y='n_syn_vis', linestyle='none', linewidth = 1,
   errorbar=('se'), 
    capsize=0.2, color="black", zorder = 2)

ax[0].set_ylim(0, 8000)
ax[1].set_ylim(0, 1500)
ax[0].set_title('Total number of input synapses', fontsize=15, y=1.1)
ax[1].set_title('Total number of V1 output synapses', fontsize=15, y=1.1)

#### Peak correlation ####

In [ ]:
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
from scipy import stats

In [ ]:
def get_peak_positions(dataset, n_points=1000):
    """Return first and second KDE peak positions."""
    if len(dataset) < 2:
        return np.nan, np.nan

    x_vals = np.linspace(dataset.min(), dataset.max(), n_points)
    kde_vals = gaussian_kde(dataset)(x_vals)
    peaks, _ = find_peaks(kde_vals)

    if len(peaks) == 0:
        return np.nan, np.nan

    peak_positions = x_vals[peaks]

    first_peak = peak_positions[0]
    second_peak = peak_positions[1] if len(peak_positions) > 1 else first_peak

    return first_peak, second_peak


In [ ]:
# Get peak positions for proofread cells
data = syn_DF_m.copy()
cells = prf_cells.query('cell_type_grouped in ["L2", "L3"]')

data_E = data[data["classification_system"] == "excitatory_neuron"]
data_I = data[data["classification_system"] == "inhibitory_neuron"]

results = []

for _, row in cells.iterrows():
    nucleus_id = row.Nucleus_ID

    E_dataset = data_E.loc[data_E["Nucleus_ID"] == nucleus_id, "pt_position_y"].values
    I_dataset = data_I.loc[data_I["Nucleus_ID"] == nucleus_id, "pt_position_y"].values

    E_peak1, E_peak2 = get_peak_positions(E_dataset)
    I_peak1, I_peak2 = get_peak_positions(I_dataset)

    results.append({
        "Nucleus_ID": nucleus_id,
        "cell_type_IT": row.cell_type_IT,
        "soma_position_y": row.pt_position_y,
        "1st_E_peak_position": E_peak1,
        "1st_I_peak_position": I_peak1,
        "sec_E_peak_position": E_peak2,
        "sec_I_peak_position": I_peak2,
    })

Output_peak = pd.DataFrame(results)

In [ ]:
# Get peak positions for column E cells
col_syn_DF = pd.read_parquet(path + 'Column-E-output-synapses-v1718.parquet')
I_types = ["ITC", "STC", "PTC", "DTC"]
IT_types = ["L2a", "L2b", "L2c", "L3a", "L3b"]

data = col_syn_DF.copy().query('id_post.notna()')
cells = mtype_col.query('cell_type in @IT_types')

data_E = data.query('cell_type_post not in @I_types')
data_I = data.query('cell_type_post in @I_types')

results = []


for _, row in cells.iterrows():
    nucleus_id = row.target_id

    E_dataset = data_E.loc[data_E["id_pre"] == nucleus_id, "pt_position_y_post"].values
    I_dataset = data_I.loc[data_I["id_pre"] == nucleus_id, "pt_position_y_post"].values

    E_peak1, E_peak2 = get_peak_positions(E_dataset)
    I_peak1, I_peak2 = get_peak_positions(I_dataset)

    results.append({
        "Nucleus_ID": nucleus_id,
        "cell_type_IT": row.cell_type,
        "soma_position_y": row.pt_position_y,
        "1st_E_peak_position": E_peak1,
        "1st_I_peak_position": I_peak1,
        "2nd_E_peak_position": E_peak2,
        "2nd_I_peak_position": I_peak2,
    })

Output_peak = pd.DataFrame(results)

In [ ]:
# data plotting, extended figure 1b
plotdata = Output_peak.query('cell_type_IT in ["L2a", "L2b", "L2c", "L3a","L3b"]') 
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

value1 = "1st_I_peak_position"
value2 = "soma_position_y"

sns.scatterplot(data=plotdata, x=value1, y=value2, ax=ax,
                edgecolor = 'black', hue='cell_type_IT', palette=m_color, 
                alpha= 0.8, s=50)

#trend line
tdata = plotdata
x2=  tdata[value1].astype(int)
y2=  tdata[value2].astype(int)
slope, intercept, r_value, p_value, std_err = stats.linregress(x2, y2)

t = np.linspace(0, 600, 100)
plt.plot(t, slope*t+intercept, color='red', linestyle='dashed', alpha=0.4, lw=1)
plt.plot(t, 1*t, color='grey', linestyle='dashed', alpha=0.7, lw=1)

ax.set_xlim(0, 500)
ax.set_ylim(0, 500)

for pos in ['top', 'right']:
    ax.spines[pos].set_visible(False)

print(f"Correlation coefficient (r): {r_value}", f"P-value: {p_value}", f"Slope: {slope}")

#### L4 IT output ####

In [ ]:
col_conn = pd.read_parquet(path + 'Column-all-output-v1718-mtype.parquet')
L4_output = pd.read_parquet(path+'Proofread_L4_output_synapses_v1718.parquet')

In [ ]:
# to get data of column or proofread L4 IT output
#data = col_conn.query('cell_type_pre in ["L4a","L4b","L4c"] & cell_type_post not in ["DTC","PTC","STC","ITC"]')
data = L4_output.query('strategy_axon == "axon_fully_extended" & cell_type_post not in ["DTC","PTC","STC","ITC"]')
data = data.sort_values(by=['cell_type_pre','pt_position_y_pre'], ascending=True)
id_order = data.id_pre.unique()
data_p = data.pivot_table(index='cell_type_post', columns='id_pre', values='syn_id', aggfunc='count').fillna(0)
data_p = data_p.reindex(columns=id_order)
data_percent = data_p/data_p.sum(axis=0) * 100

desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b']
data_percent = data_percent.reindex(desired_order)


In [ ]:
#Extended Figure 1d,e

coords = data.groupby('id_pre').agg({'cell_type_pre': 'min'}).cell_type_pre.value_counts()
coords = coords.reindex(['L4a', 'L4b', 'L4c'])
data_percent = data_percent.astype(float)
fig, ax = plt.subplots(figsize=(15, 5), dpi=300)
sns.heatmap (data_percent.iloc[:, :], cmap='viridis', vmax=30)
for i,n in enumerate(coords):
    x = coords[:i+1].sum()
    x2 = coords[:i].sum() 
    ax.vlines(x, 0, 25, color='white', lw=1.5)
    ax.text(x2+(x-x2)/2, -1.8, f'{coords.index[i]}', ha='center', va='top', fontsize=15, color='black')

In [ ]:
# Figure 2e
data_g = data.groupby(['cell_type_pre','id_pre', 'cell_type_post']).agg({'syn_id':'count'})
data_g['pct'] = data_g['syn_id'] / data_g.groupby('id_pre')['syn_id'].transform('sum') * 100
fig, ax = plt.subplots(figsize=(3, 6), dpi=300)
sns.lineplot (data=data_g.query('cell_type_post!="L6wm"'),
              x='pct', y='cell_type_post', 
              errorbar='se',
              hue='cell_type_pre', palette=m_color, marker='o', 
              ax=ax,
              orient='y',)
ax.set_yticks(range(len(desired_order)))
ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
ax.set_yticklabels(desired_order)
ax.set_xticklabels(ax.get_xticklabels(),  fontsize=12)
ax.legend(title='Presynaptic cell type', bbox_to_anchor=(1, 0.1), loc='lower right', fontsize=12)
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)

In [ ]:
#Extended Figure 1e
summary = (
    data_g
    .groupby(['cell_type_pre','cell_type_post'])['pct']
    .agg(
        mean='mean',
        se=lambda x: x.std(ddof=1) / np.sqrt(len(x))
    )
    .reset_index()
)
fig, ax = plt.subplots(figsize=(2, 4), dpi=300)
summary_pivot = summary.pivot(index='cell_type_post', columns='cell_type_pre', values='mean')
summary_pivot = summary_pivot.reindex(index=desired_order, columns=['L4a', 'L4b', 'L4c'])
summary_pivot_percent = summary_pivot/summary_pivot.sum() * 100
sns.heatmap(summary_pivot_percent, annot=False, fmt=".1f", cmap='viridis', vmax=15)

#### Motif soma location ####

In [ ]:
#Extended Figure 2d
Layers=np.array([91.80615154, 261.21908419, 396.8631847 , 537.04973966, 753.58049474])
x = Layers

fig, ax =plt.subplots(2, 8, figsize=(16,6), facecolor='white', sharey=True,dpi=300)
ax = ax.flatten()

for i, motif in enumerate(motif_df['motif'].unique().tolist()):
    motif_subset = motif_df[motif_df['motif'] == motif]
    sns.scatterplot(data=motif_subset, 
                    x='pt_position_x',
                    y='pt_position_y',
                 ax=ax[i],
                 hue='pre_cell_type',  
                 alpha=0.7,               
                 palette=m_color, 
                 edgecolor='white',
                 legend=False)
    ax[i].hlines(x, 400, 1050, color='grey', alpha=0.5, linestyle='dashed', lw=1)
    ax[i].set_xlim(400,1050)
    ax[i].set_ylim(800,0)
    ax[i].spines[['top', 'right']].set_visible(False)
    ax[i].set_title(f'Motif {motif}', fontsize=16)
    ax[i].set_axis_off()

#### Motif heatmaps ####

In [ ]:
# Figure 2f
Data = motif_df.copy() 

motif = Data.motif.unique().tolist()
cluster = Data.cluster.unique().tolist()
Data.sort_values(['cluster','pre_cell_type', 'pt_position_y'], ascending=True, inplace=True)

coords = Data.cluster.value_counts().reindex(cluster)
ct_list = list(zip(Data.index, Data.pre_cell_type))
desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC', 'ITC',]

Data_sorted = Data.reindex(columns=desired_order)
data_array=Data_sorted.iloc[:,:22].T


fig, ax = plt.subplots(figsize=(30, 10), dpi=300)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 18])


ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])


ax1.set_xlim(0, data_array.shape[1])
ax1.set_xticks([])
ax1.set_yticks([])

for id, ct in ct_list:
      ax1.vlines(data_array.columns.get_loc(id), 0, 10, color=m_color[ct], lw=6)

sns.heatmap(data_array, cmap='Spectral_r', ax=ax2, vmin=0, vmax=0.4, cbar=False) #

for i,n in enumerate(coords):
    x = coords[:i+1].sum()
    x2 = coords[:i].sum() 
    
    ax2.vlines(x, 0, 22, color='white', lw=2)
    ax2.text(x2+(x-x2)/2, -1, f'{motif[i]}', ha='center', va='top', fontsize=20, color='black')
#optional right side y-axis for cell type labels
# ax3 = ax2.twinx()  # share x-axis, independent y-axis
# ax3.set_ylim(ax2.get_ylim())  # sync limits
# ax3.set_yticks(np.arange(len(Data_sorted.columns[:22])) + 0.5)  # center ticks
# ax3.set_yticklabels(Data_sorted.columns[:22], rotation=0)
# ax3.yaxis.set_label_position("right")




#### Reciprocal plots ####

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
pre_syn = pd.read_parquet(path+'All-E-input-synapses-v1718.parquet')
post_syn = pd.read_parquet(path+'All-E-output-synapses-v1718.parquet')

In [ ]:
# for motif analysis loaded from local column files
pre_syn_DF = pre_syn.merge(motif_df[['id', 'cluster']], left_on='id_pre', right_on='id', how='inner')
pre_syn_DF_m = pre_syn_DF.merge(mtype_V1[['cell_type','target_id']], left_on='id_post', right_on='target_id', how='left', suffixes=('_motif','_mtype'))
pre_syn_DF_m.rename(columns={'syn_id': 'id_syn','target_id':'Nucleus_ID'}, inplace=True)
pre_syn_DF_m.drop(columns=['id'], inplace=True)

pre_syn_g = pre_syn_DF_m.groupby(['Nucleus_ID','pre_pt_root_id']
                                             ).agg({'id_syn':'count','size':'sum'}
                                                   ).rename(columns={'id_syn':'n_syn'}).reset_index()

post_syn_DF = post_syn.merge(motif_df[['id', 'cluster','pt_position_y','pre_cell_type']], left_on='id_post', right_on='id', how='inner')
post_syn_DF = post_syn_DF.merge(mtype_V1[['cell_type','target_id','pt_position_y']], left_on='id_pre', right_on='target_id', how='inner', suffixes=('_post','_pre'))
post_syn_DF.rename(columns={'syn_id': 'id_syn','id_pre':'Nucleus_ID','pre_cell_type':'cell_type_post', 'cell_type':'cell_type_IT'}, inplace=True)


In [ ]:
# for mtype analysis loaded from local column files
pre_syn_col = pre_syn.merge(mtype_col[['target_id','cell_type']], left_on='id_pre', right_on='target_id', how='inner')
pre_syn_col_DF_m = pre_syn_col.merge(mtype_col[['cell_type','target_id']], left_on='id_post', right_on='target_id', how='inner', suffixes=('_pre','_post'))
pre_syn_col_DF_m.rename(columns={'syn_id': 'id_syn','target_id_post':'Nucleus_ID'}, inplace=True)

pre_syn_g = pre_syn_col_DF_m.groupby(['Nucleus_ID','pre_pt_root_id']
                                             ).agg({'id_syn':'count','size':'sum'}
                                                   ).rename(columns={'id_syn':'n_syn'}).reset_index()

post_syn_DF = post_syn.merge(mtype_col[['target_id', 'cell_type','pt_position_y']], left_on='id_post', right_on='target_id', how='inner')
post_syn_DF = post_syn_DF.merge(mtype_V1[['cell_type','target_id','pt_position_y']], left_on='id_pre', right_on='target_id', how='inner', suffixes=('_post','_pre'))
post_syn_DF.rename(columns={'syn_id': 'id_syn','id_pre':'Nucleus_ID', 'cell_type_pre':'cell_type_IT'}, inplace=True)


In [ ]:
#limit analysis to proofread cells
post_syn_DF = post_syn_DF.query('Nucleus_ID in @prf_cells.Nucleus_ID')
pre_syn_g = pre_syn_g.query('Nucleus_ID in @prf_cells.Nucleus_ID')

In [ ]:
# post Motif/Col synapse matrix

post_matrix = post_syn_DF.groupby(['Nucleus_ID','post_pt_root_id']).agg( n_syn=('id_syn','size'),
         cluster=('cluster','min'),
         cell_type_mtype=('cell_type_post','min'),
         cell_type_IT=('cell_type_IT','min'),
         pre_pt_position_y=('pt_position_y_pre','min'),
         post_pt_position_y=('pt_position_y_post','min'),).reset_index()

post_pivot_df=post_matrix.set_index(['Nucleus_ID','post_pt_root_id'])['n_syn'].unstack('post_pt_root_id', fill_value=0) 

#sort the ROW presynaptic ET/IT cells by cell type
pre_order = post_matrix[['Nucleus_ID','cell_type_IT','pre_pt_position_y']].sort_values(by=['cell_type_IT','pre_pt_position_y'], ascending=True)
row_order=pre_order['Nucleus_ID'].unique().tolist()
post_pivot_df=post_pivot_df.reindex(row_order, axis=0)

#sort the COLUMN postsynaptic cells by cell type
post_order = post_matrix[['post_pt_root_id','cluster','cell_type_mtype','post_pt_position_y']].sort_values(by=['cluster', 'cell_type_mtype','post_pt_position_y'], ascending=True)
#post_order = post_matrix[['post_pt_root_id','cell_type_mtype','post_pt_position_y']].sort_values(by=['cell_type_mtype','post_pt_position_y'], ascending=True)

col_order = post_order.post_pt_root_id.unique().tolist()
post_pivot_df = post_pivot_df[col_order]

#make long table for plotting
post_melt_df = post_pivot_df.reset_index().melt(id_vars='Nucleus_ID', var_name='post_cell', value_name='n_syn')

#change this for motif/mtype 'cluster' or 'cell_type_mtype'
cell_identity = 'cluster' 
type_map = (post_matrix[['post_pt_root_id',f'{cell_identity}']]
             .drop_duplicates('post_pt_root_id')
             .set_index('post_pt_root_id')[f'{cell_identity}']) 
pre_E_map = (post_matrix[['Nucleus_ID','cell_type_IT']]
             .drop_duplicates('Nucleus_ID')
             .set_index('Nucleus_ID')['cell_type_IT'])

post_melt_df['cell_type'] = post_melt_df['post_cell'].map(type_map)
post_melt_df['cell_type_IT'] = post_melt_df['Nucleus_ID'].map(pre_E_map)    

# get the number of presynaptic synapses 

tmp = pre_syn_g.rename(columns={'pre_pt_root_id':'post_cell', 'n_syn':'n_pre_syn'})
post_melt_df = post_melt_df.merge(
    tmp[['Nucleus_ID','post_cell','n_pre_syn']],
    on=['Nucleus_ID','post_cell'],
    how='left'
)
post_melt_df['n_pre_syn'] = post_melt_df['n_pre_syn'].fillna(0).astype(int)


post_melt_df[['Nucleus_ID','post_cell']] = post_melt_df[['Nucleus_ID','post_cell']].astype(str)

In [ ]:
#change between motif/mtype using 'cluster' or 'cell_type_mtype'

coords=post_matrix.groupby(f'{cell_identity}').agg({'post_pt_root_id':'nunique'}).post_pt_root_id.values 
pre_coords = post_matrix.groupby('cell_type_IT').agg({'Nucleus_ID':'nunique'}).Nucleus_ID.values


In [ ]:
#Extended Figure 2e,f, Figure 5d
fig, ax = plt.subplots(figsize=(25,40), dpi=300)
cmap_E = sns.light_palette("#C71585", as_cmap=True)
cmap_I = sns.light_palette("teal", as_cmap=True)
divider = make_axes_locatable(ax)

ax_hstrip = divider.append_axes("top", size="2%", pad=0.03)
ax_hstrip.set_xlim(-17, 312)

ax_hstrip.set_xticks([])
ax_hstrip.set_yticks([])

for id in post_pivot_df.columns:
      ct = post_matrix.query('post_pt_root_id == @id')['cell_type_mtype'].values[0]
      ax_hstrip.vlines(post_pivot_df.columns.get_loc(id), -3, 0, color=m_color[ct], lw=9)
      
ct_list =post_melt_df.cell_type.unique()

#for plotting EtoE
#coords = coords[2:20]

for i,n in enumerate(coords):
    x = coords[:i+1].sum()-0.5
    ax.vlines(x, -1, post_pivot_df.shape[0], color='grey', lw=1.5, linestyle='dashed', zorder=10)
    x2 = coords[:i].sum() 
    text_pos = (x + x2)/2
    ax.text(text_pos, -20, f'{ct_list[i]}', ha='center', va='top', fontsize=20, color='black')

for i,n in enumerate(pre_coords):
    y = pre_coords[:i+1].sum()-0.5
    ax.hlines(y, -2, post_pivot_df.shape[1], color='grey', lw=1.5, linestyle='dashed', zorder=10)

sns.scatterplot(data = post_melt_df, #.query('cell_type not in ["PTC","DTC","ITC","STC"]'),
                ax=ax,
                x='post_cell', y='Nucleus_ID', 
                size='n_syn', 
                size_norm=(0, 10), 
                sizes=(0, 100),
                edgecolor='black', linewidth=0.8,
                hue= 'n_pre_syn', palette= cmap_I, #cmap_E
                hue_norm=(0,15),
                alpha=1,
                legend=True)


for pos in ['left','right', 'top']:
    ax.spines[pos].set_visible(False)

ax.tick_params(axis='both', which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
plt.legend(loc="upper right", bbox_to_anchor=(1, 1))


In [ ]:
post_melt_df.Nucleus_ID = post_melt_df.Nucleus_ID.astype(float)
#motif reciprocal rate by cell type
ct_list = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b']
motif_recip_df = pd.DataFrame(index = ct_list, columns= range(1,18))
for ct in ct_list:
    data = prf_cells.query(f'cell_type_IT == "{ct}"')
    ids = data.Nucleus_ID.unique().tolist()
    sub_df = post_melt_df.query('Nucleus_ID in @ids')
    total_bi=0
    total_uni=0
    for motif in range(1,18):
        n_bi=0
        n_uni=0
        sub_motif_df = sub_df.query('cell_type == @motif')
        for i, row in sub_motif_df[:].iterrows():
            if row.n_syn >0 and row.n_pre_syn >0:
                n_bi += 1
                total_bi += 1
            if row.n_syn >0 and row.n_pre_syn ==0:
                n_uni += 1
                total_uni += 1
        if n_bi + n_uni ==0:
            recip_rate = 0
        else:
            recip_rate = n_bi/(n_bi+n_uni)*100
        motif_recip_df.loc[ct, motif] = recip_rate
    total_recip_rate = total_bi/(total_bi+total_uni)*100

In [ ]:
#mtype reciprocal rate by cell type
post_melt_df.Nucleus_ID = post_melt_df.Nucleus_ID.astype(float)
ct_list = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b']
mtype_recip_df = pd.DataFrame(index = ct_list, columns = ct_list)
for ct in ct_list:
    data = prf_cells.query(f'cell_type_IT == "{ct}"')
    ids = data.Nucleus_ID.unique().tolist()
    sub_df = post_melt_df.query('Nucleus_ID in @ids')
    total_bi=0
    total_uni=0
    for mtype in ct_list:
        n_bi=0
        n_uni=0
        sub_motif_df = sub_df.query('cell_type == @mtype')
        for i, row in sub_motif_df[:].iterrows():
            if row.n_syn >0 and row.n_pre_syn >0:
                n_bi += 1
                total_bi += 1
            if row.n_syn >0 and row.n_pre_syn ==0:
                n_uni += 1
                total_uni += 1
        if n_bi + n_uni ==0:
            recip_rate = 0
        else:
            recip_rate = n_bi/(n_bi+n_uni)*100
        mtype_recip_df.loc[ct, mtype] = recip_rate
    total_recip_rate = total_bi/(total_bi+total_uni)*100

In [ ]:
# Figure 2h,i
fig, ax = plt.subplots(figsize=(16, 3), dpi=300)
sns.heatmap(motif_recip_df.astype(float), 
            ax=ax,
            cmap=cmap_I, 
            vmax=10, annot=True, 
            fmt=".1f",    
            cbar=True)

#### TC axon output ####

In [ ]:
TC_lgn = pd.read_csv(path+'prf-LGN-output-mtype-v1718.csv')

In [ ]:
#Extended Figure 1f
desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
       'L5a', 'L5b', 'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c','L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC', 'ITC',]
TC_conn_pivot = TC_lgn.pivot_table(index='pre_pt_root_id', columns='cell_type', values='syn_id', fill_value=0)
TC_conn_pivot = TC_conn_pivot.reindex(columns= desired_order)
TC_conn_percent = (TC_conn_pivot.T / TC_conn_pivot.sum(axis=1)).T * 100

Fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
sns.heatmap(TC_conn_percent.T, cmap='viridis', vmax=30)

In [ ]:
fig, ax = plt.subplots(figsize=(1, 4), dpi=300)
sns.barplot(data=TC_lgn.reset_index(), x='syn_id', y='cell_type', 
             orient='h', palette=m_color,
             errorbar='se',
             err_kws= {'linewidth': 1},
             order=desired_order)
for pos in ['top', 'right']:
    ax.spines[pos].set_visible(False)

### Figure 3 ###

#### Synapse distribution ####

In [ ]:
#Figure 3a
cell_type =["L2a","L2b","L2c"]
data = syn_DF_m.query(f'cell_type_IT in {cell_type} & cell_type in ["L5a"]') 

Layers=np.array([ 91.80615154, 261.21908419, 396.8631847 , 537.04973966,
       753.58049474])
fig, ax = plt.subplots(figsize=(8, 8),dpi=300)
sns.scatterplot(data=data[['syn_pt_position_x','syn_pt_position_y','cell_type','size']], 
                ax=ax,
                x='syn_pt_position_x', y='syn_pt_position_y', 
                hue='cell_type', palette=m_color, 
                size = 'size',
                size_norm=(0,50000), sizes=(0, 60),
                edgecolor = 'black',
                alpha=0.8)
ax.set_ylim(700,0)
ax.set_aspect("equal")
x = Layers
ax.hlines(x, 200, 1200, color='grey', alpha=0.5, linestyle='dashed', lw=1)
ax.spines[['right','top']].set_visible(False)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
ax.set_title(f'pos of {cell_type} to L5 IT synapses', fontsize=10, y=1.05, loc='center')

In [ ]:
#Figure 3a
cell_type =["L2a","L2b","L2c"]
data = syn_DF_m.query(f'cell_type_IT in {cell_type} & cell_type in ["L5a","L5b","L5ET"]')# 

fig, ax = plt.subplots(figsize=(2, 4), dpi=300)
sns.histplot(data=data, y='syn_pt_position_y',
             ax=ax,
             stat='count',
            hue='cell_type', palette=m_color,
            element='step', fill=False,
            binwidth=10, kde=True, alpha=0,
            legend=False)
ax.set_ylim(700,0)
x = Layers
ax.hlines(x, 0, 220, color='grey', alpha=0.5, linestyle='dashed', lw=1)
ax.spines[['top', 'right']].set_visible(False)

#### Synapse shuffle ####

In [ ]:
import random

In [ ]:

def spatial_shuffle_n_times(shuffle_all_possibilities_df, n_shuffles):

    # this is supposed to take the shuffled df, then group bby 
    # pre_pt_root_id and then by real synapse id

    grouped_df = shuffle_all_possibilities_df.groupby(['real_synapse_id'])
    # now for each group, go through and select a single row from that group
    # then append that to a list
    all_shuffled_data = []
    for name, group in grouped_df:
        
        group.reset_index(drop = True, inplace = True)
        i = random.choices(range(len(group)), k = n_shuffles)
        all_shuffled_data.append(group.iloc[i].values)

    return pd.DataFrame(np.vstack(all_shuffled_data), columns = shuffle_all_possibilities_df.columns)

In [ ]:
# Shuffle from acquired synapse list and calculate the ratio of true vs shuffled connectivity
# !!! uploading shuffled synapse files for individual cells in progress !!!
data = prf_cells

E_subtypes = {'Subtype':['L2a','L2b','L2c','L3a','L3b','L4a','L4b','L4c','L5a', 'L5b','L5ET',
                         'L5NP','L6tall-a','L6tall-b','L6tall-c','L6short-a','L6short-b']}
I_subtypes = {'Subtype':['DTC','PTC','STC','ITC']} 

E_Syn_DF = pd.DataFrame(E_subtypes)
E_Cell_DF = pd.DataFrame(E_subtypes)
I_Syn_DF = pd.DataFrame(I_subtypes)
I_Cell_DF = pd.DataFrame(I_subtypes)


real_E_Syn_DF = pd.DataFrame(E_subtypes)
real_E_Cell_DF = pd.DataFrame(E_subtypes)
real_I_Syn_DF = pd.DataFrame(I_subtypes)
real_I_Cell_DF = pd.DataFrame(I_subtypes)


for i, row in data[:].iterrows():
    #df = pd.read_csv(path3+'analysis_results/spatial shuffle/'+str(row.Nucleus_ID)+'_spatial_shuffle_ids.csv', index_col=0)
    df = df.merge(mtype_V1[['id_ref','classification_system', 'cell_type']], left_on = 'real_post_nuc_id', right_on = 'id_ref', how = 'inner')
    df = df.merge(mtype_V1[['id_ref','classification_system', 'cell_type']], left_on = 'shuffled_post_nuc_id', right_on = 'id_ref', how = 'inner', suffixes=('_real', '_shuffled'))

    df_E = df.query('classification_system_real == "excitatory_neuron" & classification_system_shuffled == "excitatory_neuron"')
    df_I = df.query('classification_system_real == "inhibitory_neuron" & classification_system_shuffled == "inhibitory_neuron"')

    E_shuffle_df = spatial_shuffle_n_times(df_E, n_shuffles = 1000)
    I_shuffle_df = spatial_shuffle_n_times(df_I, n_shuffles = 1000)

    E_shuffle_syn_mtype = E_shuffle_df.groupby('cell_type_shuffled').agg({'shuffled_synapse_id': 'count'})
    E_shuffle_syn_mtype = E_shuffle_syn_mtype.rename(columns={'shuffled_synapse_id': row.Nucleus_ID})

    I_shuffle_syn_mtype = I_shuffle_df.groupby('cell_type_shuffled').agg({'shuffled_synapse_id': 'count'})
    I_shuffle_syn_mtype = I_shuffle_syn_mtype.rename(columns={'shuffled_synapse_id': row.Nucleus_ID})

    E_Syn_DF=pd.merge(E_Syn_DF, E_shuffle_syn_mtype, left_on='Subtype', right_on='cell_type_shuffled', how='left') 
    I_Syn_DF=pd.merge(I_Syn_DF, I_shuffle_syn_mtype, left_on='Subtype', right_on='cell_type_shuffled', how='left')

    E_real_syn_mtype = df_E.groupby('cell_type_real').agg({'real_synapse_id': 'nunique'})
    E_real_syn_mtype = E_real_syn_mtype.rename(columns={'real_synapse_id': row.Nucleus_ID})
    I_real_syn_mtype = df_I.groupby('cell_type_real').agg({'real_synapse_id': 'nunique'})
    I_real_syn_mtype = I_real_syn_mtype.rename(columns={'real_synapse_id': row.Nucleus_ID})


    real_E_Syn_DF=pd.merge(real_E_Syn_DF, E_real_syn_mtype, left_on='Subtype', right_on='cell_type_real', how='left')
    real_I_Syn_DF=pd.merge(real_I_Syn_DF, I_real_syn_mtype, left_on='Subtype', right_on='cell_type_real', how='left')
    

    
E_Syn_DF.fillna(0, inplace=True)
I_Syn_DF.fillna(0, inplace=True)

real_E_Syn_DF.fillna(0, inplace=True)
real_I_Syn_DF.fillna(0, inplace=True)

E_Syn_DF.set_index('Subtype', inplace=True)
I_Syn_DF.set_index('Subtype', inplace=True)
real_E_Syn_DF.set_index('Subtype', inplace=True)
real_I_Syn_DF.set_index('Subtype', inplace=True)

E_ratio = real_E_Syn_DF/E_Syn_DF*1000
I_ratio = real_I_Syn_DF/I_Syn_DF*1000

all_ratio = pd.concat([E_ratio, I_ratio], axis=0)

In [ ]:
#Figure 3c plot True vs. shuffled synapse ratio by cell type
data = prf_cells
all_ratio = pd.read_csv(path + 'spatial_shuffle_ratio_v1718.csv', index_col=0)
all_ratio.columns = all_ratio.columns.astype(np.float64)

fig, ax = plt.subplots(3, 5, figsize=(20, 20), sharey=True, dpi=300)
ax = ax.flatten()
for i, (type, axis) in enumerate(zip(data.cell_type_IT.unique(), ax)): 
    subset_id = data.query('cell_type_IT == @type')['Nucleus_ID']
    subset = all_ratio.loc[:, subset_id]
    sns.barplot(data = subset.T, ax=axis, orient = 'h', errorbar='se', palette = m_color, alpha=0.7)
    sns.swarmplot(data=subset.T, ax=axis, orient= 'h',  palette=m_color, size=4.5, alpha=1)
    axis.set_title(type, fontsize=16)
    axis.set_xlabel('True vs. shuffled synapse', fontsize=14)
    axis.set_ylabel('mtype', fontsize=18)
    axis.set_xlim(0, 3)
    for pos in ['top', 'right']:
        axis.spines[pos].set_visible(False)
    
plt.tight_layout()

### Figure 4 and Extended Figure 4 ###

#### L5 IT typing ####

In [ ]:
L5_feat = pd.read_csv(path + 'L5_IT_lda_umap_features.csv', index_col=0)

In [ ]:
cluster_to_type = {
    0: "L5b-2",
    1: "L5a",
    2: "L5b-1",
}

L5_feat["L5_type"] = L5_feat["cluster"].map(cluster_to_type)

In [ ]:
#Clipped scaler 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.base import BaseEstimator, TransformerMixin

class PercentileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower=0.1, upper=99.9):
        self.lower = lower
        self.upper = upper
        self.lower_bounds_ = None
        self.upper_bounds_ = None
    
    def fit(self, X, y=None):
        self.lower_bounds_ = np.percentile(X, self.lower, axis=0)
        self.upper_bounds_ = np.percentile(X, self.upper, axis=0)
        return self
    
    def transform(self, X):
        return np.clip(X, self.lower_bounds_, self.upper_bounds_)

def make_clipped_scaler(lower=0.1, upper=99.9) -> Pipeline:
    """ Create a data preprocessing pipeline that first scales the data using RobustScaler
    and then clips the values to specified percentiles.

    Parameters
    ----------
    lower: float
        The lower percentile for clipping.
    upper: float
        The upper percentile for clipping.

    Returns
    -------
    Pipeline
        A sklearn Pipeline object that performs scaling and clipping.
    """
    return Pipeline([
        ('scaler', RobustScaler()),
        ('clipper', PercentileClipper(lower=lower, upper=upper)),
    ])

##### UMAP #####

In [ ]:
# Figure 4b UMAP plot
f, ax =plt.subplots(figsize=(7,7), facecolor='white', dpi=300)

sns.scatterplot(data=L5_feat,  
                x='umap0_l5', y='umap1_l5', ax =ax, s=20, alpha=0.8,  
                palette='tab10', edgecolor='white', 
                color='lightgrey',
                legend = True)
sns.scatterplot(data=L5_feat.query('cell_type_IT != "L5b-s"'),  
                x='umap0_l5', y='umap1_l5', ax =ax, s=80, alpha=1,  
                palette=m_color, edgecolor='white', 
                hue='cell_type_IT',
                hue_order=['L5a', 'L5b-1', 'L5b-2'],
                legend = True)

for pos in ['right', 'top', ]:
    ax.spines[pos].set_visible(False)

ax.set_xlabel('UMAP 1', fontsize=20)
ax.set_ylabel('UMAP 2', fontsize=20)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=15)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=15)

plt.legend(bbox_to_anchor=(1, 0.2),loc='right', fontsize=15)

##### LDA #####

In [ ]:
#Figure 4f  LDA plot
cluster_cmap = {0: "#BD7CC8", 1: "#663399", 2: "#F506D1"}

f, ax =plt.subplots(figsize=(9,7), facecolor='white', dpi=300)

sns.scatterplot(data=L5_feat,  
                x='lda_x', y='lda_y', ax =ax, s=20, alpha=0.3,  
                palette=cluster_cmap, edgecolor='white', hue='cluster',
                legend = False)
sns.scatterplot(data=L5_feat.query('cell_type_IT != "L5b-s"'),
                x='lda_x', y='lda_y', ax =ax, s=35, alpha=1,
                palette=m_color, edgecolor='black', 
                hue='cell_type_IT',
                hue_order=['L5a', 'L5b-1', 'L5b-2'])
ax.set_xlim(-20, 20)
ax.set_ylim(-15, 15)
for pos in ['right', 'top', ]:
    ax.spines[pos].set_visible(False)
ax.set_xlabel('LDA 1', fontsize=20)
ax.set_ylabel('LDA 2', fontsize=20)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=15)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=15)
plt.legend(bbox_to_anchor=(1, 0.185),loc='right', fontsize=15)

##### Heatmap #####

In [ ]:
#ClippedScaler
features = L5_feat.iloc[:, 3:101].values
my_scaler =  make_clipped_scaler()
scaled_features = my_scaler.fit_transform(features)

In [ ]:
#Or StandardScaler
from sklearn.preprocessing import StandardScaler
features = L5_feat.iloc[:, 3:101].values
scaled_features = StandardScaler().fit_transform(features)

In [ ]:
L5_DF = L5_feat.copy()

feature_cols =L5_DF.columns[3:101]
L5_DF[feature_cols] = scaled_features

cluster_order = [1, 2, 0]
L5_DF["cluster"] = pd.Categorical(
    L5_DF["cluster"],
    categories=cluster_order,
    ordered=True,
)
L5_DF = L5_DF.sort_values(["cluster", "soma_depth"])

In [ ]:
#Figure 4g
data = L5_DF.iloc[:, 3:101]
coords = L5_DF.cluster.value_counts().reindex(cluster_order)
fig, ax = plt.subplots(figsize=(35,20), facecolor='white', dpi=300)
sns.heatmap(data.T, 
            ax=ax,
            cmap='RdBu_r', 
            vmin=-1, vmax=1, cbar=True)
for i,n in enumerate(coords):
    x = coords[:i+1].sum()
    ax.vlines(x, 0, 100, color='white', lw=4)

##### Soma depth #####

In [ ]:
fig, ax = plt.subplots(figsize=(1.5,3), facecolor='white', dpi=300)
sns.violinplot(data=L5_DF, 
               x='L5_type', y='soma_depth', 
               ax=ax, 
               hue='L5_type', 
               palette=m_color, 
               linewidth=0.5,
               legend=False)
ax.set_ylim(600, 200)
for pos in ['top', 'right']:
    ax.spines[pos].set_visible(False)

#### L23 -> L5 connectivity ####

In [ ]:
source_types = ["L2a", "L2b", "L2c", "L3a", "L3b", "L5ET"]

L23_conn = syn_DF_m.loc[syn_DF_m["cell_type_IT"].isin(source_types)]

L23_conn_ct = pd.merge(
    L23_conn,
    L5_feat[["cell_id", "L5_type"]],
    left_on="target_id",
    right_on="cell_id",
    how="inner",
)

type_weights = {
    "L5a": 0.27,
    "L5b-1": 0.44,
    "L5b-2": 0.29,
}

L23_conn_g = (
    L23_conn_ct
    .groupby(["cell_type_IT", "Nucleus_ID", "L5_type"], as_index=False)
    .agg(id_syn=("id_syn", "count"))
)

L23_conn_g["weight"] = L23_conn_g["L5_type"].map(type_weights)
L23_conn_g["weighted_syn"] = L23_conn_g["id_syn"] / L23_conn_g["weight"]

group_cols = ["cell_type_IT", "Nucleus_ID"]

L23_conn_g["pct_syn"] = (
    L23_conn_g["id_syn"]
    / L23_conn_g.groupby(group_cols)["id_syn"].transform("sum")
    * 100
)

L23_conn_g["pct_weighted_syn"] = (
    L23_conn_g["weighted_syn"]
    / L23_conn_g.groupby(group_cols)["weighted_syn"].transform("sum")
    * 100
)

In [ ]:
#Figure 4i, j
fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
sns.barplot(data=L23_conn_g, 
            ax=ax,
            x='cell_type_IT',
            y='pct_weighted_syn', 
            hue='L5_type', 
            hue_order=['L5a','L5b-1','L5b-2'],
            errorbar='se',
            palette=m_color,
            dodge=True,
                )
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)
ax.set_ylabel('Percentage of synapses (%)', fontsize=15, fontname='Arial')
ax.set_xlabel('Presynaptic cell type', fontsize=15, fontname='Arial')
ax.set_xticklabels(ax.get_xticklabels(), fontsize=15, fontname='Arial')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=15, fontname='Arial')
ax.set_ylim(0, 70)
plt.legend(bbox_to_anchor=(1.25, 0.2),loc='right', fontsize=15)

#### L23 percentage #### 

In [ ]:
L5_L23_p = pd.read_csv(path + 'L5IT-L23-synapse-percentage.csv')

In [ ]:
#Extended Figure 4a
L5_L23_p.sort_values(by='cell_type_IT', ascending=True, inplace=True)

fig, ax = plt.subplots(figsize=(3, 3), dpi=300)
sns.swarmplot(data=L5_L23_p, x='cell_type_IT', y='L23_Percentage', 
              ax=ax,
              hue='cell_type_IT',
            palette=m_color)
sns.pointplot(ax = ax,
    data=L5_L23_p, x='cell_type_IT', y='L23_Percentage', linestyle='none', linewidth = 1,
   errorbar=('se'), 
    capsize=0.2, color="black", zorder = 100)
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)
ax.set_ylim(0, 100)

#### EI ratio ####

In [ ]:
#Extended Figure 4c
data_pivot = syn_DF_m.pivot_table(index=['Nucleus_ID','cell_type_IT'], 
                                columns='classification_system',
                                values='id_syn', 
                                aggfunc={'id_syn':'count'}).fillna(0)
data_pivot['EI_ratio'] = data_pivot['excitatory_neuron'] / data_pivot['inhibitory_neuron']
data_pivot.reset_index(inplace=True)
data_pivot.sort_values(by='cell_type_IT', ascending=True, inplace=True)
data_sub = data_pivot.query('cell_type_IT in ["L5a", "L5b-1", "L5b-2","L5b-s"]')


fig,ax = plt.subplots(figsize=(3, 4), dpi=300)
sns.barplot(data=data_sub, ax=ax, x='cell_type_IT', y='EI_ratio',
            hue='cell_type_IT', palette=m_color,
            errorbar='se', alpha=0.7, linewidth=1)
sns.swarmplot(data=data_sub, 
            ax=ax, x='cell_type_IT', y='EI_ratio',
            hue='cell_type_IT', palette=m_color, 
            size=4,
            alpha=1,
            linewidth=0.5,
            # dodge=True,
            )
for pos in ['right', 'top']:
    ax.spines[pos].set_visible(False)
ax.set_ylim(0, 5)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=12)

#### E-E Co-targeting ####

In [ ]:
syn_DF_m_grouped = syn_DF_m.copy().groupby(['Nucleus_ID','id_ref']).agg({'cell_type_IT':'min', 'cell_type':'min'}).reset_index()

In [ ]:
# individual cell pair analysis
master_df =pd.DataFrame(columns=['Nucleus_ID_1','Nucleus_ID_2','cell_type_IT_1','cell_type_IT_2','L2a','L2b','L2c','L3a','L3b','L4a','L4b','L4c','L5a', 'L5b','L5ET',
                         'L5NP','L6tall-a','L6tall-b','L6tall-c','L6short-a','L6short-b', 'DTC','PTC','STC', 'ITC'])
x=0
for i, row1 in prf_cells[:].iterrows():
    sub_df1 = syn_DF_m_grouped.query('Nucleus_ID == @row1.Nucleus_ID')
    for j, row2 in prf_cells[:].iterrows():
        if row1.Nucleus_ID == row2.Nucleus_ID:
            continue
        
        co_target_df = pd.DataFrame(index=['L2a','L2b','L2c','L3a','L3b','L4a','L4b','L4c','L5a', 'L5b','L5ET',
                         'L5NP','L6tall-a','L6tall-b','L6tall-c','L6short-a','L6short-b', 'DTC','PTC','STC', 'ITC'])
        sub_df2 = syn_DF_m_grouped.query('Nucleus_ID == @row2.Nucleus_ID')

        overlap = pd.merge(sub_df1, sub_df2, on='id_ref', how='inner', suffixes=('_1','_2'))
        combined = pd.merge(sub_df1, sub_df2, on='id_ref', how='outer', suffixes=('_1','_2'))
        combined['merged_type'] = combined['cell_type_1'].fillna(combined['cell_type_2'])

        overlap = overlap.groupby(['cell_type_1']).agg({'id_ref':'count'}).rename(columns={'id_ref':'overlap count'})
       
        combined = combined.groupby(['merged_type']).agg({'id_ref':'count'}).rename(columns={'id_ref':'combined count'})

        co_target_df = pd.merge(co_target_df, overlap, left_index=True, right_index=True, how='left')
        co_target_df = pd.merge(co_target_df, combined, left_index=True, right_index=True, how='left')
        co_target_df.fillna(0, inplace=True)

        co_target_rate = co_target_df['overlap count']/co_target_df['combined count']

        new_row = [row1.Nucleus_ID, row2.Nucleus_ID, row1.cell_type_IT, row2.cell_type_IT] + co_target_rate.tolist()
        master_df.loc[x] = new_row
       
        x+=1
master_df.fillna(0, inplace=True)

In [ ]:
query_list1 = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b'] 
query_list2 = ['L5a', 'L5b-1','L5b-2', 'L5ET'] 
query_list = query_list1 + query_list2
ct_list = ['DTC','PTC','STC', 'ITC'] 
Data = master_df.query('cell_type_IT_1 in @query_list & cell_type_IT_2 in @query_list')

fig, ax = plt.subplots(1,4, figsize=(20,3.5), dpi=300) 
ax = ax.flatten()
for i in range(4):
    ct = ct_list[i]

    #data = master_df.pivot_table(index='Nucleus_ID_1', columns='Nucleus_ID_2', values='L2a') #.query('cell_type_IT_1=="L2a" & cell_type_IT_2=="L3a"')
    data = Data.pivot_table(index='cell_type_IT_1', columns='cell_type_IT_2', values=ct, aggfunc='mean') 
    #data = Data.pivot_table(index='Nucleus_ID_1', columns='Nucleus_ID_2', values=ct)
    data_norm = data*100 #-Data[ct].mean()
    mask = np.triu(np.ones_like(data_norm, dtype=bool), k=1)
    sns.heatmap(data_norm, ax=ax[i], cmap=cmap_I, vmin=0, vmax=10, 
                mask = mask,
                cbar=False,cbar_kws={'location': 'bottom', 'shrink': 0.8, 'pad':0.3, 'label': 'Co-targeted rate (%)'},
                annot=True, 
                annot_kws={
                "size": 8,},
                fmt='.2f', 
) 
    ax[i].set_title(f'{ct}', fontsize=25, fontname='Arial', y=1.1)
    ax[i].set_xlabel('Cell type 2', fontsize=15, fontname='Arial')
    ax[i].set_ylabel('Cell type 1', fontsize=15, fontname='Arial')
    ax[i].set_xticklabels(ax[i].get_xticklabels(), fontsize=10, rotation=0, fontname='Arial')
    ax[i].set_yticklabels(ax[i].get_yticklabels(), rotation=0, horizontalalignment='right', fontsize=10, fontname='Arial')

plt.subplots_adjust(wspace=0.5, hspace=0.7)

### Figure 5 ###

#### E-I Co-targeting ####

In [ ]:
visp_cotarget_mean = pd.read_csv(path + 'co_target_mean_all_E_target_v1718.csv', index_col=0)
col_cotarget_mean = pd.read_csv(path + 'co_target_mean_column_E_target_v1718.csv', index_col=0)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4), dpi=300)
sns.heatmap(col_cotarget_mean, 
            #annot=True, fmt='.1f',
            linecolor='white', 
            linewidth=1.5,
            cmap='Purples', 
            vmin=0,
            vmax=100,
            cbar_kws={'label': 'Co-targeting rate'}, ax=ax)

#### L5 motif output ####

In [ ]:
motif_df.sort_values(by=['cluster', 'pt_position_y'], inplace=True)
m8_PTC = motif_df.query('motif == "8/10" & pre_cell_type == "PTC"')
m9a = motif_df.query('motif == "9a" & pre_cell_type == "PTC"')
m9b = motif_df.query('motif == "9b" & pre_cell_type == "DTC"')
all_pr_I_conn = all_pr_I_conn.merge(mtype_V1[['id_ref','cell_type']], left_on = 'post_nuc_id', right_on = 'id_ref', how = 'inner')

In [ ]:
conn_df = all_pr_I_conn.query('pre_nuc_id in @m8_PTC.id')
conn_df_g = conn_df.groupby(['pre_nuc_id','cell_type']).agg({'n_syn':'sum'})
desired_order = ['L2a', 'L2b', 'L2c', 'L3a', 'L3b', 'L4a', 'L4b', 'L4c',
      'L5a', 'L5b',  'L5ET', 'L5NP', 'L6tall-a',
       'L6tall-b', 'L6tall-c', 'L6short-a', 'L6short-b', 'DTC', 'PTC', 'STC','ITC']
id_order = m8_PTC.id.tolist()
df_p = conn_df_g.pivot_table(index='cell_type', columns='pre_nuc_id', values='n_syn', aggfunc='sum', fill_value=0)
df_p = df_p.reindex(desired_order, columns=id_order)


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(4, 4),  dpi=300)
sns.heatmap(df_p, cmap='viridis', vmax=1000, cbar=False, ax=ax[0])
for pos in ['top', 'right']:
    ax[0].spines[pos].set_visible(False)
    ax[1].spines[pos].set_visible(False)

sns.barplot(data=df_p.T, ax=ax[1],
            palette=m_color,
            orient='h',
            errorbar='se',
            err_kws={'linewidth': 1},
)
ax[1].set_xlim(0, 900)
ax[1].set_ylabel("")
ax[1].tick_params(axis='y', labelleft=False)
fig.subplots_adjust(wspace=0.05)

#### L5 cell soma depth distribution ####

In [ ]:
L5a_pos = prf_cells.query('cell_type_IT == "L5a" ').pt_position_y.tolist()
L5b1_pos = prf_cells.query('cell_type_IT == "L5b-1" ').pt_position_y.tolist()
L5b2_pos = prf_cells.query('cell_type_IT == "L5b-2" ').pt_position_y.tolist()
L5ET_pos = prf_cells.query('cell_type_IT == "L5ET" ').pt_position_y.tolist()

motif8_pos = motif_df.query('motif == "8/10" & pre_cell_type == "PTC"').pt_position_y.tolist()
motif9a_pos = motif_df.query('motif == "9a"& pre_cell_type == "PTC"').pt_position_y.tolist()
motif9b_pos = motif_df.query('motif == "9b"& pre_cell_type == "DTC"').pt_position_y.tolist()
E_positions = {'L5a': L5a_pos, 'L5b-1': L5b1_pos, 'L5b-2': L5b2_pos, 'L5ET': L5ET_pos}
I_positions = {'Motif 8': motif8_pos, 'Motif 9a': motif9a_pos, 'Motif 9b': motif9b_pos}

In [ ]:
E_df = pd.concat(
    [pd.DataFrame({"group": k, "position": np.asarray(v)}) for k, v in E_positions.items()],
    ignore_index=True
).dropna()

I_df = pd.concat(
    [pd.DataFrame({"group": k, "position": np.asarray(v)}) for k, v in I_positions.items()],
    ignore_index=True
).dropna()


fig, ax = plt.subplots(1, 2, figsize=(6, 6), dpi=300, sharey=True)

sns.kdeplot(
    data=E_df,
    y="position",
    hue="group",
    ax=ax[0],
    fill=False,
    common_norm=False,
    palette=m_color,
    legend=False,
)

sns.kdeplot(
    data=I_df,
    y="position",
    hue="group",
    ax=ax[1],
    fill=False,
    alpha=1,
    common_norm=False,
    linestyle="dashed",
    palette={
        "Motif 8": (0.4752, 0.34, 0.86),
        "Motif 9a": (0.7872, 0.34, 0.86),
        "Motif 9b": (0.86, 0.34, 0.6208),
    },
    legend=False,
)

# mirror left panel
ax[0].invert_xaxis()

# depth range
ax[0].set_ylim(700, 300)
ax[1].set_ylim(700, 300)

# make panels further apart
fig.subplots_adjust(wspace=0.3)

# labels
ax[0].set_xlabel("Density")
ax[1].set_xlabel("Density")
ax[0].set_ylabel("Position")
ax[1].set_ylabel("")